# Thai sarcasm on a free cloud GPU

รัน task `thai_sarcasm` (ชุด gold 127 ข้อ) กับโมเดลเปิดภาษาไทยเก่งๆ (Typhoon / SeaLLM / OpenThaiGPT / Qwen)
บน GPU ฟรีของ Colab แล้วเก็บเป็นตาราง "$0 open-model frontier" — ต่อยอดแกน cost ของโปรเจกต์

**ก่อนรัน:** เมนู Runtime -> Change runtime type -> **GPU (T4)**

ใช้ task/scorer ตัวเดียวกับในเครื่อง (`Gold/lm_eval/`) เป๊ะ -> ผลเทียบกับ GPT/WangchanBERTa ได้ตรง

### 1) ติดตั้ง

In [ ]:
!pip -q install "lm-eval==0.4.12" accelerate datasets bitsandbytes


### 2) (ถ้าจำเป็น) ล็อกอิน Hugging Face
บางโมเดล (ฐาน Llama เช่น Typhoon/OpenThaiGPT) ต้องกด accept license บนหน้าโมเดล + ใส่ token ก่อน
ข้ามเซลล์นี้ได้ถ้าใช้แค่โมเดลเปิด (เช่น Qwen)

In [ ]:
from huggingface_hub import login
login()   # วาง HF token จาก huggingface.co/settings/tokens


### 3) อัปโหลด `gold.csv`
เลือกไฟล์ `Gold/gold.csv` จากเครื่องคุณ (127 ข้อ ป้ายคนจริง)

In [ ]:
from google.colab import files
up = files.upload()   # เลือก gold.csv


### 4) เขียนไฟล์ task + scorer (คัดลอกจาก `Gold/lm_eval/` ตรงๆ)

In [ ]:
import os
os.makedirs('thai_task', exist_ok=True)   # %%writefile ไม่สร้างโฟลเดอร์ให้ ต้องสร้างก่อน


In [ ]:
%%writefile thai_task/thai_sarcasm.yaml
# lm-evaluation-harness custom task: Thai sarcasm (ประชด) detection on the gold set
# ประเมินโมเดล "เปิด" (Typhoon/SeaLLM/OpenThaiGPT/Qwen/Llama) บน gold 127 ข้อ ด้วยต้นทุน API = $0
#
# แบบ multiple_choice: โมเดลให้ loglikelihood ของ "ใช่/ไม่ใช่" -> เลือกอันมากกว่า
#   (แนวเดียวกับวิธี logprob-threshold ของ GPT ในโปรเจกต์ แต่ไม่ต้อง generate -> เร็ว/ทำซ้ำได้ทุกโมเดล)
#
# รันจากโฟลเดอร์ Gold/ (เพื่อให้ data_files: gold.csv หาไฟล์เจอ):
#   lm_eval --model hf --model_args pretrained=<thai-model> \
#           --include_path lm_eval --tasks thai_sarcasm --num_fewshot 0 \
#           --log_samples --output_path lm_eval_out
# แล้วคิด F1/precision/recall + เขียน CSV ให้ตรงฟอร์แมตโปรเจกต์ด้วย:
#   python lm_eval/score_lm_eval.py --samples lm_eval_out --out <model>_lmeval_pred.csv
task: thai_sarcasm
dataset_path: csv
dataset_kwargs:
  data_files:
    test: gold.csv
test_split: test
output_type: multiple_choice
process_docs: !function utils.process_docs
doc_to_text: !function utils.doc_to_text
doc_to_choice: ["ไม่ใช่", "ใช่"]      # index 0 = ไม่ประชด, index 1 = ประชด
doc_to_target: gold                    # int 0/1 (เติมโดย process_docs)
target_delimiter: " "
metric_list:
  # acc = ตัวเลขเร็วๆ จาก harness เอง; F1/precision/recall (positive class) คิดจาก score_lm_eval.py
  # (จงใจไม่ผูก F1 ไว้ใน YAML เพราะ metric API ต่างกันตามเวอร์ชัน harness -- ใช้สคริปต์เป็น source of truth)
  - metric: acc
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0


In [ ]:
%%writefile thai_task/utils.py
# -*- coding: utf-8 -*-
"""helper สำหรับ task thai_sarcasm ใน lm-evaluation-harness

- process_docs : เติมคอลัมน์ gold (int 0/1) จาก label (str)
- doc_to_text  : ประกอบ prompt = rubric + ข้อความ (rubric ชุดเดียวกับ DETECT_SYS ใน predict.py)
"""
import datasets

# ตรงกับ DETECT_SYS ใน Gold/predict.py (ให้เทียบกับผล GPT ได้ยุติธรรม)
RUBRIC = (
    'ตัดสินว่าข้อความภาษาไทยนี้ "ประชด/เสียดสี" หรือไม่\n'
    "ประชด = เจตนาจริงตรงข้ามกับความหมายผิวเผิน เพื่อเหน็บหรือแสดงความไม่พอใจ"
)


def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:
    def _add_gold(doc):
        doc["gold"] = 1 if str(doc.get("label", "")).strip() in ("1", "1.0") else 0
        doc["text"] = str(doc.get("text", "")).strip()
        return doc
    return dataset.map(_add_gold)


def doc_to_text(doc) -> str:
    return f"{RUBRIC}\n\nข้อความ: {doc['text']}\nเป็นการประชดหรือไม่? คำตอบ:"


In [ ]:
%%writefile score_lm_eval.py
# -*- coding: utf-8 -*-
"""แปลงผล lm-evaluation-harness (task thai_sarcasm) -> F1/precision/recall + CSV ฟอร์แมตโปรเจกต์

harness เก็บ per-item ไว้ใน samples_*.jsonl (ต้องรันด้วย --log_samples --output_path ...)
สคริปต์นี้อ่าน loglikelihood ของสองตัวเลือก [ไม่ใช่, ใช่] -> P(ประชด)=softmax(...)[1] -> label
แล้วเขียน CSV ที่มี pred_prob / pred_label / pred_decision เหมือน predict.py และ batch_eval.py
-> เอาไปเข้า compare_systems.py / bootstrap / McNemar ได้เลย (เทียบข้ามทุกระบบได้)

ใช้:
  python lm_eval/score_lm_eval.py --samples lm_eval_out --out typhoon_lmeval_pred.csv
  python lm_eval/score_lm_eval.py --samples lm_eval_out/.../samples_thai_sarcasm_xx.jsonl --out o.csv --threshold 0.5
"""
import argparse
import glob
import json
import math
import os
import sys

sys.stdout.reconfigure(encoding="utf-8")


def find_samples(path):
    if os.path.isdir(path):
        hits = glob.glob(os.path.join(path, "**", "samples_thai_sarcasm_*.jsonl"), recursive=True)
        if not hits:
            sys.exit(f"ไม่พบ samples_thai_sarcasm_*.jsonl ใต้ {path} (รัน harness ด้วย --log_samples --output_path ยัง?)")
        return max(hits, key=os.path.getmtime)   # อันใหม่สุด
    return path


def choice_logliks(rec):
    """ดึง loglikelihood ต่อ choice จาก record (รองรับหลายเวอร์ชันของ harness)"""
    r = rec.get("filtered_resps") or rec.get("resps") or []
    out = []
    for c in r:
        x = c
        while isinstance(x, (list, tuple)) and x:
            x = x[0]
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return []
    return out


def gold_of(rec):
    t = rec.get("target")
    if isinstance(t, int):
        return t
    doc = rec.get("doc") or {}
    if "gold" in doc:
        return int(doc["gold"])
    return 1 if str(t).strip() in ("1", "ใช่") else 0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--samples", required=True, help="โฟลเดอร์ output_path หรือไฟล์ samples_*.jsonl")
    ap.add_argument("--out", required=True, help="ไฟล์ CSV ผลลัพธ์")
    ap.add_argument("--threshold", type=float, default=None,
                    help="ตัด P(ประชด) ที่ค่านี้ (ไม่ใส่ = argmax loglik ปกติ)")
    a = ap.parse_args()

    path = find_samples(a.samples)
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        sys.exit(f"อ่านไม่เจอ record ใน {path}")

    import pandas as pd
    recs, yt, yp = [], [], []
    for rec in rows:
        lls = choice_logliks(rec)
        doc = rec.get("doc") or {}
        text = str(doc.get("text", "")).replace("\n", " ")
        g = gold_of(rec)
        if len(lls) < 2:
            recs.append({"text": text, "label": g, "pred_prob": None,
                         "pred_label": None, "pred_decision": "error"})
            continue
        m = max(lls[0], lls[1])
        p1 = math.exp(lls[1] - m) / (math.exp(lls[0] - m) + math.exp(lls[1] - m))  # P(ใช่/ประชด)
        pred = (1 if p1 >= a.threshold else 0) if a.threshold is not None else (1 if lls[1] > lls[0] else 0)
        recs.append({"text": text, "label": g, "pred_prob": round(p1, 3),
                     "pred_label": pred, "pred_decision": "sarcasm" if pred else "not_sarcasm"})
        yt.append(g); yp.append(pred)

    pd.DataFrame(recs).to_csv(a.out, index=False, encoding="utf-8-sig")

    TP = sum(t == 1 and p == 1 for t, p in zip(yt, yp))
    FP = sum(t == 0 and p == 1 for t, p in zip(yt, yp))
    FN = sum(t == 1 and p == 0 for t, p in zip(yt, yp))
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    print(f"อ่าน {path}")
    print(f"เขียน {a.out} · n={len(yt)} · positives(true)={sum(yt)} · predicted_pos={sum(yp)}")
    print(f"precision={P:.3f}  recall={R:.3f}  F1={F1:.3f}")


if __name__ == "__main__":
    main()


### 5) รันทุกโมเดล + เก็บผลเป็นตาราง frontier
แต่ละโมเดลจะดาวน์โหลด (7-8B ~15GB) แล้วรัน 127 ข้อบน GPU — ตัวละไม่กี่นาที
โมเดลไหน gated/OOM จะถูกข้ามไป ไม่ล้มทั้ง loop

In [ ]:
import subprocess, os, pandas as pd

# แก้/เพิ่มโมเดลได้ตามใจ (ต้องเป็น causal LM บน HF Hub)
#   Qwen = เปิด ไม่ต้อง login ; อีก 3 ตัว (ฐาน Llama) มักต้อง accept license + login (เซลล์ 2)
MODELS = {
    "qwen2.5-7b":     "Qwen/Qwen2.5-7B-Instruct",
    "typhoon-1.5-8b": "scb10x/llama-3-typhoon-v1.5-8b-instruct",
    "seallm-7b":      "SeaLLMs/SeaLLM-7B-v2.5",
    "openthaigpt-7b": "openthaigpt/openthaigpt-1.0.0-7b-chat",
}
# T4 ฟรีมี 16GB -> โหลด 4-bit กัน OOM (7-8B แบบ fp16 มักเต็ม) ; bitsandbytes ติดตั้งแล้วในเซลล์ 1
QUANT = "load_in_4bit=True"     # อยากได้ความแม่นเต็มและมี GPU ใหญ่: เปลี่ยนเป็น "dtype=float16"

def run_one(tag, mid):
    out = f"out_{tag}"
    cmd = ["lm_eval", "--model", "hf",
           "--model_args", f"pretrained={mid},{QUANT}",
           "--device", "cuda", "--batch_size", "8",
           "--include_path", "thai_task", "--tasks", "thai_sarcasm",
           "--num_fewshot", "0", "--log_samples", "--output_path", out]
    print("="*60, "\nrunning", mid, f"({QUANT})")
    if subprocess.run(cmd).returncode != 0:
        print("!!", mid, "failed -> skip  (ดู error ที่สตรีมด้านบน: gated ต้อง login เซลล์ 2 / OOM / อื่นๆ)")
        return None
    subprocess.run(["python", "score_lm_eval.py", "--samples", out, "--out", f"{tag}_pred.csv"])
    return f"{tag}_pred.csv"

rows = []
for tag, mid in MODELS.items():
    f = run_one(tag, mid)
    if not f or not os.path.exists(f):
        continue
    d = pd.read_csv(f)
    yt = d["label"].astype(str).str.strip().isin(["1", "1.0"]).astype(int)
    yp = d["pred_label"].astype(str).str.strip().isin(["1", "1.0"]).astype(int)
    TP = int(((yt==1)&(yp==1)).sum()); FP = int(((yt==0)&(yp==1)).sum()); FN = int(((yt==1)&(yp==0)).sum())
    P = TP/(TP+FP) if TP+FP else 0; R = TP/(TP+FN) if TP+FN else 0; F1 = 2*P*R/(P+R) if P+R else 0
    rows.append({"model": tag, "F1": round(F1,3), "precision": round(P,3),
                 "recall": round(R,3), "pred_pos": int(yp.sum())})
    print(f"  -> {tag}: F1 {F1:.3f} | P {P:.3f} | R {R:.3f}")

if not rows:
    print("\n!! ไม่มีโมเดลไหนรันผ่านเลย -- เลื่อนขึ้นไปดูเหตุผลของแต่ละตัว")
    print("   ส่วนใหญ่คือ: (1) โมเดล gated -> รันเซลล์ 2 (login) ก่อน  (2) OOM -> QUANT ควรเป็น load_in_4bit=True")
    print("   ลองทีละตัวก่อน เริ่มจาก Qwen (เปิด ไม่ต้อง login)")
else:
    tbl = pd.DataFrame(rows).sort_values("F1", ascending=False)
    print("\n=== $0 open-model frontier (Thai sarcasm, gold 127) ===")
    print(tbl.to_string(index=False))
    tbl.to_csv("open_model_frontier.csv", index=False)


### 6) เทียบกับระบบในโปรเจกต์ (ตัวเลขอ้างอิง)

| ระบบ | F1 | ต้นทุน |
|---|---|---|
| GPT bot (gpt-4.1-mini + threshold) | ~0.72 | ~\$0.0001/ข้อ |
| WangchanBERTa (fine-tuned) | ~0.62 | \$0 (ต้องเทรน) |
| โมเดลเล็กบน M1 (Qwen 1.5B) | ~0.36 | \$0 |

คำถาม: โมเดลเปิดภาษาไทยตัวใหญ่ (แถวนี้) ไล่ทัน GPT ที่ \$0 ได้ไหม? ตารางด้านบนตอบ

### 7) โหลดผลกลับเครื่อง

In [ ]:
# โหลดผลกลับเครื่อง (prediction CSV ฟอร์แมตเดียวกับ predict.py -> เข้า compare_systems.py ได้)
from google.colab import files
import glob
files.download("open_model_frontier.csv")
for f in glob.glob("*_pred.csv"):
    files.download(f)
